# Tool-Discovery Agent

Discover bioinformatics tools from papers, inject them into your agent, and run tasks.

**Flow:** Scan agent -> Inject tools -> Run tasks via downstream agent

In [ ]:
import os, sys, subprocess
ST2_DIR = '/content/st2'
if os.path.isdir(os.path.join(ST2_DIR, '.git')):
    subprocess.run(['git', '-C', ST2_DIR, 'pull', '--ff-only'], capture_output=True)
else:
    if os.path.exists(ST2_DIR):
        subprocess.run(['rm', '-rf', ST2_DIR], check=True)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caixiaoyao2025/st2.git', ST2_DIR], check=True)
ST2_DIR = '/content/st2'
sys.path.insert(0, ST2_DIR)
os.chdir(ST2_DIR)
# Demo input so the default task runs through without the user
# supplying a file. seqmagick is pure-Python/pip-installable, so it
# works in Colab; bqtools would need a Rust/cargo toolchain.
_DEMO_FASTA = ('>seq1 example\n'
              'ACGTACGTACGTACGTACGT\n'
              '>seq2 example\n'
              'TTGGCCTAAGGCCTTAGGCA\n'
              '>seq3 example\n'
              'GATTACAGATTACAGATTACA\n')
with open('reads.fasta', 'w') as _f:
    _f.write(_DEMO_FASTA)
print('demo reads.fasta written:', len(_DEMO_FASTA), 'bytes')
!pip install -q langchain langchain-openai langgraph pydantic graph-tool-call 2>/dev/null || true
from IPython.display import display, Markdown
display(Markdown('**Setup done**'))

## Step 1 - Input your agent & API key

In [ ]:
import os, subprocess
from IPython.display import display, Markdown
import ipywidgets as widgets

# ---- Known agent mappings: method + register method ----
KNOWN_AGENTS = {
    'biomni': {'method': 'go', 'register': 'add_tool'},
    'cellagent': {'method': 'run', 'register': 'add_tool'},
    'geneagent': {'method': 'run', 'register': 'add_tool'},
    'crispr': {'method': 'run', 'register': 'add_tool'},
    'biochatter': {'method': 'run', 'register': 'add_tool'},
    'langchain': {'method': 'invoke', 'register': 'add_tool'},
    'smolagents': {'method': 'run', 'register': 'add_tool'},
    'dspy': {'method': 'forward', 'register': 'add_tool'},
    'crewai': {'method': 'kickoff', 'register': 'add_tool'},
    'metagpt': {'method': 'run', 'register': 'add_tool'},
}
PROBE_ORDER = ['go', 'run', 'execute', 'predict', 'forward', 'invoke']

# ---- Agent presets: pick from the dropdown to fill the URL and auto-connect ----
AGENT_PRESETS = {
    'Biomni A1 (snap-stanford/Biomni)': 'https://github.com/snap-stanford/Biomni.git',
    'BioChatter (biocypher/biochatter)': 'https://github.com/biocypher/biochatter.git',
    'smolagents (huggingface)': 'https://github.com/huggingface/smolagents.git',
    'DSPy (stanfordnlp)': 'https://github.com/stanfordnlp/dspy.git',
    'CrewAI': 'https://github.com/crewAIInc/crewAI.git',
    'MetaGPT': 'https://github.com/geekan/MetaGPT.git',
}
agent_dropdown = widgets.Dropdown(
    options=['(custom URL)'] + list(AGENT_PRESETS.keys()),
    value='Biomni A1 (snap-stanford/Biomni)',
    description='Agent:', layout=widgets.Layout(width='90%'))

path_input = widgets.Text(
    value='https://github.com/snap-stanford/Biomni.git',
    placeholder='Git URL or /content/my_agent',
    description='Agent:', layout=widgets.Layout(width='90%'))
key_input = widgets.Password(
    value='sk-CXVEKD43upOkHWTdq1RJP3SMC4OyspQOkqB4ymqw6IJazWyB', placeholder='ark-... or sk-...',
    description='API Key:', layout=widgets.Layout(width='90%'))
model_input = widgets.Text(
    value='minimax-m3',
    description='Model:', layout=widgets.Layout(width='70%'))
base_input = widgets.Text(
    value='https://tokenhub.tencentmaas.com/v1',
    description='Base URL:', layout=widgets.Layout(width='90%'))
btn = widgets.Button(description='Connect agent', button_style='primary')
out = widgets.Output()

def on_connect(_):
    out.clear_output()
    with out:
        agent_src = path_input.value.strip()
        api_key = key_input.value.strip()
        model = model_input.value.strip()
        base_url = base_input.value.strip()
        if not agent_src:
            display(Markdown('**ERROR:** Enter agent path or URL')); return
        if not api_key:
            display(Markdown('**ERROR:** Enter API key')); return
        if agent_src.startswith('http'):
            agent_dir = '/content/_agent_' + agent_src.split('/')[-1].replace('.git', '')
            if not os.path.isdir(agent_dir):
                subprocess.run(['git', 'clone', '--depth', '1', agent_src, agent_dir],
                               capture_output=True, text=True)
            display(Markdown(f'Cloned to `{agent_dir}`'))
        else:
            agent_dir = agent_src
            if not os.path.isdir(agent_dir):
                display(Markdown(f'**ERROR:** `{agent_dir}` not found')); return
        os.environ['OPENAI_API_KEY'] = api_key
        os.environ['OPENAI_BASE_URL'] = base_url
        os.environ['OPENAI_MODEL'] = model
        os.environ['WESTLAKE_API_KEY'] = api_key
        os.environ['BIOMNI_SOURCE'] = 'Custom'
        os.environ['BIOMNI_LLM'] = model
        os.environ['BIOMNI_CUSTOM_BASE_URL'] = base_url
        os.environ['BIOMNI_CUSTOM_API_KEY'] = api_key
        get_ipython().user_ns['agent_dir'] = agent_dir
        get_ipython().user_ns['api_key_val'] = api_key
        get_ipython().user_ns['model_val'] = model
        get_ipython().user_ns['base_url_val'] = base_url
        display(Markdown(f'**Agent:** `{agent_dir}` | **Model:** `{model}` | **Key:** `{api_key[:8]}...`'))

def _on_preset_change(change):
    if change.get('name') != 'value' or change.get('new') == '(custom URL)':
        return
    url = AGENT_PRESETS.get(change['new'])
    if url:
        path_input.value = url
        on_connect(None)

agent_dropdown.observe(_on_preset_change)
btn.on_click(on_connect)
display(widgets.VBox([agent_dropdown, path_input, key_input, model_input, base_input, btn, out]))

## Step 2 - Scan agent & detect wiring

In [ ]:
import os, sys, json
from IPython.display import display, Markdown
import yaml

from agent_connector.scanner import build_schema

schema = build_schema(agent_dir, include_evidence=False)
detected = [
    '**Detected:**',
    '- agent_class = `' + str(schema.get('agent_class', 'N/A')) + '`',
    '- module_path = `' + str(schema.get('module_path', 'N/A')) + '`',
    '- registration_method = `' + str(schema.get('registration_method', 'N/A')) + '`',
    '- wiring_style = `' + str(schema.get('wiring_style', 'N/A')) + '`',
    '- init_signature = `' + str(schema.get('init_signature', 'N/A')) + '`',
]
display(Markdown('<br>'.join(detected)))

reg_path = os.path.join(ST2_DIR, 'data', 'mcp_registry.yaml')
tools = yaml.safe_load(open(reg_path, encoding='utf-8'))['tools']
display(Markdown(f'**Registry:** {len(tools)} tools'))

## Step 3 - Resolve execution method

In [ ]:
from IPython.display import display, Markdown

agent_class = (schema.get('agent_class') or '').lower()

# Layer 1: known agent -> direct mapping
exec_method = None
for pat, info in KNOWN_AGENTS.items():
    if pat in agent_class or pat in agent_dir.lower():
        exec_method = info['method']
        display(Markdown(f'**Known agent** `{pat}` -> `{exec_method}`'))
        break

# Layer 2: scan source for .go()/.run() hints
if exec_method is None:
    hits = {}
    for root, dirs, files in os.walk(agent_dir):
        dirs[:] = [d for d in dirs if d not in ('.git','__pycache__','node_modules','.venv')]
        for f in files:
            if f.endswith('.py'):
                try:
                    text = open(os.path.join(root,f), encoding='utf-8', errors='replace').read()
                except: continue
                for m in PROBE_ORDER:
                    hits[m] = hits.get(m, 0) + text.count(f'.{m}(')
    if any(v > 0 for v in hits.values()):
        exec_method = max(hits, key=hits.get)
        display(Markdown(f'**Source hint** -> `{exec_method}` ({hits[exec_method]} occurrences)'))

# Layer 3: default
if exec_method is None:
    exec_method = 'run'
    display(Markdown('**No signal found.** Defaulting to `run`.'))

get_ipython().user_ns['exec_method_override'] = exec_method

## Step 4 - Preflight check & install

In [ ]:
import os, sys, subprocess as _sp
from IPython.display import display, Markdown
from agent_connector.agent_preflight import preflight, preflight_report

# Auto-install missing agent runtime deps (Layer 1) detected by the preflight.
pf = preflight(agent_dir, agent_module_path=schema.get('module_path'), install_missing=True)

# Most OpenAI-compatible agents (incl. BioChatter) also need the LLM backend at
# import time. Layer 2 tool-deps are NOT auto-installed (some are import-tracer
# artifacts like 'base'/'web' that are not real packages), so install the
# known-safe LLM-backend subset explicitly.
#
# For the MCP-native path (BioChatter connects to our server.py as an MCP client)
# we also need langchain-mcp-adapters + nest_asyncio.
for _pkg in ['openai', 'langchain-openai', 'pydantic', 'langchain-community',
             'langchain-mcp-adapters', 'nest_asyncio']:
    try:
        _r = _sp.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                     capture_output=True, text=True, timeout=600)
        if _r.returncode != 0:
            print(f'  (pip install {_pkg} note: {_r.stderr.strip()[:200]})')
    except Exception as _e:
        print(f'  pip install {_pkg} failed: {_e}')

# BioChatter reads its OpenAI-compatible endpoint from GENERIC_TEST_OPENAI_BASE_URL.
_mod = (schema.get('module_path') or '').lower()
if 'biochatter' in agent_dir.lower() or 'biochatter' in _mod:
    os.environ['GENERIC_TEST_OPENAI_BASE_URL'] = os.environ.get('OPENAI_BASE_URL', '')
    display(Markdown('**BioChatter env:** `GENERIC_TEST_OPENAI_BASE_URL` set to the OpenAI base URL.'))

display(Markdown(preflight_report(pf)))


In [ ]:
import subprocess, os, sys
from IPython.display import display, Markdown

if pf.status == 'SETUP_REQUIRED':
    pkgs = pf.pip_installable
    if pkgs:
        display(Markdown(f'Installing **{len(pkgs)}** packages...'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        display(Markdown(f'**Done** (returncode={r.returncode})'))
    else:
        display(Markdown('No auto-installable packages'))
else:
    display(Markdown(f'Status = `{pf.status}`'))

## Step 5 - Create agent & inject tools

In [ ]:
import os, sys, json
from IPython.display import display, Markdown
from agent_connector.agent_runner import build_runtime, run_agent
from agent_connector.generator import generate_wiring, load_adapter

# 1) Instantiate the downstream agent (via generator adapter, or A1 fallback).
agent = None
adapter = None
_framework = (schema.get('tool_interface') or {}).get('framework')
if _framework == 'biochatter':
    # BioChatter uses an explicit startup contract: its runnable Conversation is
    # built by BioChatterAdapter.create_agent() inside build_runtime, NOT the generic
    # generate_wiring path (which would build a bare tool registry with only add_tool).
    agent = None
elif schema.get('module_path') or schema.get('registration_method'):
    try:
        _wiring = generate_wiring(tools, schema, out_dir=os.path.join(ST2_DIR, 'wiring'))
        _adapter_path = _wiring['artifacts'].get('adapter')
        adapter = load_adapter(schema.get('agent_class') or 'Agent', adapter_path=_adapter_path) if _adapter_path else None
        agent = adapter.create_agent(use_tool_retriever=False)
        display(Markdown(f'**Agent created via adapter:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'`create_agent()` failed: `{e}`'))

if agent is None and 'biomni' in agent_dir.lower():
    try:
        sys.path.insert(0, agent_dir)
        from biomni.agent.a1 import A1
        agent = A1(
            path=os.path.join(ST2_DIR, '_agent_data'),
            llm=os.environ.get('OPENAI_MODEL', 'minimax-m3'),
            source='Custom',
            base_url=os.environ.get('OPENAI_BASE_URL'),
            api_key=os.environ.get('OPENAI_API_KEY'),
            use_tool_retriever=False,
            expected_data_lake_files=[],
        )
        display(Markdown(f'**A1 agent created:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'A1 fallback failed: `{e}`'))

if agent is None:
    display(Markdown('**No native agent created** -- cannot drive tools without a downstream agent.'))

# 2) Inject discovered tools via the capability-selected Adapter.
# 3) The agent then drives its OWN planner/tool-loop via run_agent(agent, prompt).
agent, _runtime_info = build_runtime(
    agent=agent, schema=schema, tools=tools, agent_dir=agent_dir,
    model=os.environ.get('OPENAI_MODEL') or 'minimax-m3',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY') or os.environ.get('WESTLAKE_API_KEY'),
)
get_ipython().user_ns['agent'] = agent
get_ipython().user_ns['runtime_info'] = _runtime_info
display(Markdown(f"**Runtime:** adapter=`{_runtime_info.get('adapter')}` | driver=`{_runtime_info.get('driver')}`"))
if _runtime_info.get('inject_error'):
    display(Markdown(f"**Inject error:** `{_runtime_info['inject_error']}`"))
display(Markdown('**Note:** tools injected via adapter; `agent.go(prompt)` drives them (no planner loop from us).'))


## Step 5b - Manual agent init (only if Step 5 failed)

If Step 5 succeeded, **skip this cell**.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets

already_ok = is_known
if not already_ok:
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent): already_ok = True; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            already_ok = True; break

if already_ok:
    display(Markdown('Step 5 succeeded. **Skipping.**'))
else:
    display(Markdown('### Agent execution method not recognized\n\n'
        '**Tried:** ' + ', '.join(f'`{m}`' for m in PROBE_ORDER) + '\n\n'
        '**Please paste your agent init code below:**'))
    code_area = widgets.Textarea(
        value='# from my_agent import MyAgent\n# agent = MyAgent(model="gpt-4")\n# agent.add_tools(wrappers)\n',
        placeholder='agent = MyAgent(...)',
        layout=widgets.Layout(width='90%', height='150px'))
    method_input = widgets.Dropdown(
        options=['run', 'execute', 'go', 'predict', 'forward', 'invoke', '__call__'],
        value='run', description='Exec method:', layout=widgets.Layout(width='60%'))
    apply_btn = widgets.Button(description='Apply', button_style='primary')
    out = widgets.Output()

    def on_apply(_):
        out.clear_output()
        with out:
            try:
                local_ns = {'wrappers': wrappers, 'agent_dir': agent_dir}
                exec(code_area.value, local_ns)
                new_agent = local_ns.get('agent')
                if new_agent is None:
                    display(Markdown('Error: no `agent` variable found.')); return
                method = method_input.value
                fn = getattr(new_agent, method, None) if method != '__call__' else new_agent
                if fn is None or not callable(fn):
                    display(Markdown(f'Error: `{method}` not callable.')); return
                get_ipython().user_ns['agent'] = new_agent
                get_ipython().user_ns['exec_method_override'] = method
                agent = new_agent
                display(Markdown(f'**Done!** Agent=`{type(new_agent).__name__}` method=`{method}`'))
            except Exception as e:
                display(Markdown(f'Error: `{e}`'))

    apply_btn.on_click(on_apply)
    display(widgets.VBox([code_area, method_input, apply_btn, out]))

## Step 6 - Run tasks via downstream agent

Your query is sent to the agent via `agent.{method}(query)`. The agent uses the injected tools internally.

**Default task:** summarize a FASTA file - this uses `seqmagick_info` (pure-Python, pip-installable, no Rust needed) and runs out of the box in Colab. `bqtools` BINSEQ tools are also discovered, but they require a Rust/`cargo` toolchain; if selected, the runner reports them as unavailable with their install contract instead of failing silently.

In [ ]:
import time, os
from IPython.display import display, Markdown
from agent_connector.agent_runner import run_agent
import ipywidgets as widgets

query_input = widgets.Text(
    value='I have a FASTA file reads.fasta. Summarize reads.fasta: report the number of sequences and the total length of all sequences.',
    placeholder='Ask a bioinformatics question...',
    description='Query:', layout=widgets.Layout(width='95%'))
run_btn = widgets.Button(description='Run agent', button_style='success')
result_out = widgets.Output()

# ---- File upload: saved into ./uploads so tools can read them by name ----
upload = widgets.FileUpload(accept='*', multiple=True, description='Upload:')
upload_out = widgets.Output()

def _on_upload(change):
    with upload_out:
        upload_out.clear_output()
        _d = os.path.join('.', 'uploads')
        os.makedirs(_d, exist_ok=True)
        for _fname, _buf in change['new'].items():
            _data = _buf['content'] if isinstance(_buf, dict) and 'content' in _buf else _buf
            _p = os.path.join(_d, _fname)
            with open(_p, 'wb') as _f:
                _f.write(bytes(_data))
            display(Markdown(f'- saved `{_p}`'))
        if change['new']:
            display(Markdown('Uploaded files are available in your query by name, e.g. `uploads/' + _fname + '`.'))

upload.observe(_on_upload, names='value')

def run_query(_):
    result_out.clear_output()
    with result_out:
        query = query_input.value.strip()
        if not query:
            display(Markdown('**Enter a query**')); return
        display(Markdown(f'**Query:** {query}'))
        display(Markdown('**Working...**'))
        t0 = time.time()
        try:
            # Drive the downstream agent's OWN planner/tool-loop. No loop from us.
            final = run_agent(agent, query)
            elapsed = time.time() - t0
            display(Markdown(f'**Done** ({elapsed:.1f}s)'))
            display(Markdown(f'### Result\n{str(final)[:4000]}'))
        except Exception as e:
            display(Markdown(f'**Error:** `{type(e).__name__}: {e}`'))

run_btn.on_click(run_query)
display(widgets.VBox([query_input, run_btn, upload, upload_out, result_out]))
